In [ ]:
import pandas as pd


class Icd9Encoder(BaseEstimator, TransformerMixin):

    def __init__(self, comorbidity_df=None):
        self.comorbidity_df = comorbidity_df  # safe, sklearn cloneable

    def fit(self, X, y):
        df = X.copy().reset_index(drop = True)
        y = y.reset_index(drop = True)
        df['target'] = y.values

        com = self.comorbidity_df   # ACCESS HERE SAFELY

        merged = df[['subject_id','hadm_id','target']].merge(
            com, on=['subject_id','hadm_id'], how='left'
        )

        self.icd_mortality_ = merged.groupby('icd9_code')['target'].mean()
        self.global_mean_ = df['target'].mean()
        return self

    def transform(self, X):
        df = X.copy().reset_index(drop = True)
        com = self.comorbidity_df

        merged = df[['subject_id','hadm_id']].merge(
            com, on=['subject_id','hadm_id'], how='left'
        )

        merged['mortality_proxy'] = merged['icd9_code'].map(self.icd_mortality_)
        merged['mortality_proxy'] = merged['mortality_proxy'].fillna(self.global_mean_)

        agg = merged.groupby(['subject_id','hadm_id']).agg(
            max_mortality=('mortality_proxy','max'),
            mean_mortality=('mortality_proxy','mean'),
            count_comorbidities=('icd9_code','count')
        ).reset_index()

        agg = agg.fillna({
            'max_mortality': self.global_mean_,
            'mean_mortality': self.global_mean_,
            'count_comorbidities': 0
        })

        return df.merge(agg, on=['subject_id','hadm_id'], how='left').reset_index(drop = True)

class IndexResetter(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X.reset_index(drop=True)


In [1]:


test_col = "a"
monkeypatch.setattr(fe, "ICD9_DIAGNOSIS", test_col)

result = change_feature_names(df.copy())

assert result.loc[result['id'] == 1, "a"].iloc[0] == "323", "change_features: patient 1 wrong ICD9"
assert result.loc[result['id'] == 2, "a"].iloc[0] == "409", "change_features: patient 2 wrong ICD9"
assert result.loc[result['id'] == 3, "a"].iloc[0] == "TEE", "change_features: patient 3 wrong ICD9"
assert result.loc[result['id'] == 4, "a"].iloc[0].isna() == True, "change_features: patient 4 wrong ICD9"

IndentationError: unexpected indent (2444280031.py, line 8)

In [10]:
df = pd.DataFrame(
        {
            "FEATURE1": [1,2,3,4,5],
            "FEAT_2": [2,3,4,5,6],
            "ICD_FEATURE": ["44323", "54345", "grgrg", "432dw", None]

        }
    )

In [14]:
df.loc[df['FEATURE1'] == 1, 'ICD_FEATURE']

0    44323
Name: ICD_FEATURE, dtype: object

In [39]:
!source ../.venv/bin/activate

Python(72089) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [2]:
import sys
sys.executable


'/Users/gnlm/Desktop/ds_projects/Probability-of-Death-backup/.venv/bin/python'

In [1]:
import pandas as pd
import numpy as np
from probability_of_death.preprocessing.preprocessor import (
    drop_features,
    change_feature_names,
    change_comorbidities_icd9code)
from probability_of_death.feature_engineering.feature_engineering import (
    create_basic_features,
    encoder_demographics,
    encoder_icd9_codes,
    apply_icd9_mapping

)
from probability_of_death.config import DROP_FEATURES, ID_FEATURES, TRAIN_NUMERICAL_FEATURES, TRAIN_CATEGORICAL_FEATURES, TARGET

In [36]:
def random_dates(start, end, n):
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)

    return start + pd.to_timedelta(
        np.random.randint(0, (end - start).days, n), unit="D"
    )


def make_fake_patient_data(n=10):
    base_dates = pd.date_range("2020-01-01", periods=n, freq="D")
    rand_seconds = np.random.randint(0, 24*60*60, size=n)
    admit_times = base_dates + pd.to_timedelta(rand_seconds, unit="s")

    df = pd.DataFrame({
        'DOD': np.arange(n),
        'DISCHTIME': np.arange(n),
        'DEATHTIME': np.arange(n),
        'LOS': np.arange(n),
        "subject_id": np.arange(n),
        "hadm_id": np.arange(100, 100 + n),
        "ADMITTIME": admit_times,
        "RELIGION": ["NONE", "CATHOLIC", None, "JEWISH", "HINDU", "NOT SPECIFIED", "CATHOLIC", None, "UNOBTAINABLE", "HINDU"],
        "MARITAL_STATUS": ["UNKNOWN (DEFAULT)", None, "MARRIED", "DIVORCED", "WIDOWED", "LAF", None, "UNOBTAINABLE", "DIVORCED", "UNKNOWN (DEFAULT)"],
        "ETHNICITY": ["UNABLE TO OBTAIN", "WHITE", None, "BLACK", "HISPANIC", "UNABLE TO OBTAIN", "UNKNOWN/NOT SPECIFIED", None, "UNKNOWN/NOT SPECIFIED", "HISPANIC"],
        "INSURANCE": ["YES", "NO", "YES", "poe", "YES", "NO", "YES", "poe", "YES", "NO"],
        "GENDER": ["Male", None, "Female", "Declined", "None", "Male", None, "Female", None, "Declined)"],
        "ICD9_diagnosis": ["44323", "54345", None, "AB", "004", 2234, "54345", None, 4566, 23],
        "HOSPITAL_EXPIRE_FLAG": [1,0,0,0,1,1,0,0,1,0],
        "icustay_id": np.random.randint(10000,99999,size = n)
        # any required default columns
    })

    return df

def generate_fake_icd_codes(n):
    base_codes = ["44323", "54345", "25000", "41401", "V3000", "AB123"]
    return np.random.choice(base_codes, size=n)

def make_comorbidities_data(n_patients=10, codes_per_patient=3):
    # total rows = n_patients * codes_per_patient
    total = n_patients * codes_per_patient

    subject_ids = np.repeat(np.arange(n_patients), codes_per_patient)
    hadm_ids = np.repeat(np.arange(100, 100 + n_patients), codes_per_patient)

    df = pd.DataFrame({
        "SUBJECT_ID": subject_ids,
        "HADM_ID": hadm_ids,
        "SEQ_NUM": np.tile(np.arange(codes_per_patient), n_patients),
        "ICD9_CODE": generate_fake_icd_codes(total)
    })

    return df


In [43]:
test_df = make_fake_patient_data()
test_df["DOB"] = random_dates("1920-01-01", "2000-12-31", 10)
test_comorbidities_df = make_comorbidities_data()

In [44]:
def preprocess(df, comorbidities_df, icd9_mapping:None):
    df = create_basic_features(df)
    df = encoder_demographics(df)
    df = drop_features(df, DROP_FEATURES)
    df = change_feature_names(df)
    comorbidities_df = change_comorbidities_icd9code(comorbidities_df)
    df,mapping = encoder_icd9_codes(df, comorbidities_df)

    df = df.drop(ID_FEATURES, axis = 1)

    return df, mapping

In [45]:
test_df, test_mapping = preprocess(test_df, test_comorbidities_df, None)

/Users/gnlm/Desktop/ds_projects/Probability-of-Death-backup/src/probability_of_death/feature_engineering/feature_engineering.py:75: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df[MAX_MORTALITY] = train_df[MAX_MORTALITY].fillna(0)
/Users/gnlm/Desktop/ds_projects/Probability-of-Death-backup/src/probability_of_death/feature_engineering/feature_engineering.py:76: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pand

In [47]:
"HOSPITAL_EXPIRE_FLAG" in test_df.columns

True

In [2]:
train = pd.read_csv("/Users/gnlm/Desktop/ds_projects/Probability-of-Death-backup/data/mimic_train.csv")
train.head()

,HOSPITAL_EXPIRE_FLAG,subject_id,hadm_id,icustay_id,HeartRate_Min,HeartRate_Max,HeartRate_Mean,SysBP_Min,SysBP_Max,SysBP_Mean,...,Diff,ADMISSION_TYPE,INSURANCE,RELIGION,MARITAL_STATUS,ETHNICITY,DIAGNOSIS,ICD9_diagnosis,FIRST_CAREUNIT,LOS
0,0,55440,195768,228357,89.0,145.0,121.043478,74.0,127.0,106.586957,...,-61961.78470,EMERGENCY,Medicare,PROTESTANT QUAKER,SINGLE,WHITE,GASTROINTESTINAL BLEED,5789,MICU,4.5761
1,0,76908,126136,221004,63.0,110.0,79.117647,89.0,121.0,106.733333,...,-43146.18378,EMERGENCY,Private,UNOBTAINABLE,MARRIED,WHITE,ESOPHAGEAL FOOD IMPACTION,53013,MICU,0.7582
2,0,95798,136645,296315,81.0,98.0,91.689655,88.0,138.0,112.785714,...,-42009.96157,EMERGENCY,Medicare,PROTESTANT QUAKER,SEPARATED,BLACK/AFRICAN AMERICAN,UPPER GI BLEED,56983,MICU,3.7626
3,0,40708,102505,245557,76.0,128.0,98.857143,84.0,135.0,106.972973,...,-43585.37922,ELECTIVE,Medicare,NOT SPECIFIED,WIDOWED,WHITE,HIATAL HERNIA/SDA,5533,SICU,3.8734
4,0,28424,127337,225281,NaN,NaN,NaN,NaN,NaN,NaN,...,-50271.76602,EMERGENCY,Medicare,JEWISH,WIDOWED,WHITE,ABDOMINAL PAIN,56211,TSICU,5.8654


In [10]:
0 <= train['HOSPITAL_EXPIRE_FLAG'].all() <=1

np.True_

In [12]:
comor = pd.read_csv('/Users/gnlm/Desktop/ds_projects/Probability-of-Death-backup/data/extra_data/MIMIC_diagnoses.csv')
comor.columns

Index(['SUBJECT_ID', 'HADM_ID', 'SEQ_NUM', 'ICD9_CODE'], dtype='object')

In [38]:
comor

,SUBJECT_ID,HADM_ID,SEQ_NUM,ICD9_CODE
0,256,108811,1.0,53240
1,256,108811,2.0,41071
2,256,108811,3.0,53560
3,256,108811,4.0,40390
4,256,108811,5.0,5859
...,...,...,...,...
651042,65535,178280,5.0,5119
651043,65535,178280,6.0,5990
651044,65535,178280,7.0,0414
651045,65535,178280,8.0,25000


In [45]:
from src.probability_of_death.config import (TRAIN_NUMERICAL_FEATURES,
                                         TRAIN_CATEGORICAL_FEATURES,
                                         DIAGNOSIS,
                                         TARGET,
                                         ICD9_DIAGNOSIS,
                                             ID_FEATURES,
                                             SUBJECT_ID,
                                             HADM_ID,
                                             ICD9_CODE
                                             # C_SUBJECT_ID,
                                             # C_HADM_ID,
                                             # C_SEQ_NUM,
                                             # C_ICD9_CODE
)

C_SUBJECT_ID ='SUBJECT_ID'
C_HADM_ID = 'HADM_ID'
C_SEQ_NUM = 'SEQ_NUM'
C_ICD9_CODE = 'ICD9_CODE'

In [51]:
def make_synthetic_train_data(n = 50, seed = 4201):

    np.random.seed(seed)

    df = pd.DataFrame()

    for feature in TRAIN_NUMERICAL_FEATURES:
        df[feature] = np.random.randn(n)

    for feature in TRAIN_CATEGORICAL_FEATURES:
        df[feature] = np.random.choice(["A", "B", "C", "D", "E"], n)

    for feature in ID_FEATURES:
        df[feature] = np.random.randint(10000, 99999, size = n)

    # df[ICD9_DIAGNOSIS] = np.random.choice(["25000", "4019", "4280", "41401", "2724", "V4581", "51882",
    #                                     "25120", "4019aw", "42820", "41kl401", "272234", "V458ed1", "RE51881"], n)


    df[TARGET] = np.random.choice([0,1], n)
    df[DIAGNOSIS] = np.random.choice(["A123", "B34", "C123", None], n)

    return df

In [39]:
def make_synthetic_comorbidity_data(train_df, rows_per_patient = 3, seed = 4201): # Create id variables that exist in the train data to check merging
    np.random.seed(seed)

    rows = []

    icd9_codes = (
        train_df[ICD9_DIAGNOSIS].dropna().astype(str).unique().tolist()
    )
    noise_codes = ["25000", "4019", "4280", "41401", "2724", "V4581", "51881"]
    all_prefixes = list(set(icd9_codes + noise_codes))
    for _, row in train_df.iterrows():
        subj = row[SUBJECT_ID]
        hadm = row[HADM_ID]

        for seq in range(rows_per_patient):
            prefix = np.random.choice(all_prefixes)
            # icd_full_code = prefix + str(np.random.randint(10, 99))  # add some suffix

            rows.append({
                C_SUBJECT_ID : subj,
                C_HADM_ID : hadm,
                C_SEQ_NUM: seq + 1,
                C_ICD9_CODE: prefix
            })

    return pd.DataFrame(rows)

In [ ]:
C_SUBJECT_ID ='SUBJECT_ID'
C_HADM_ID = 'HADM_ID'
C_SEQ_NUM = 'SEQ_NUM'
C_ICD9_CODE = 'ICD9_CODE'

In [53]:
d = make_synthetic_train_data()
d

,HeartRate_Min,HeartRate_Max,HeartRate_Mean,SysBP_Min,SysBP_Max,SysBP_Mean,DiasBP_Min,DiasBP_Max,DiasBP_Mean,MeanBP_Min,...,count_comorbidities,AGE_AT_ADMIT,ADMISSION_TYPE,FIRST_CAREUNIT,DIAGNOSIS,QUART_DAY,subject_id,hadm_id,icustay_id,HOSPITAL_EXPIRE_FLAG
0,-0.695006,-0.980101,-0.699556,0.720019,-0.341656,-0.252636,-1.855268,0.512276,0.385961,-1.132804,...,-2.666457,1.252293,A,C,B34,D,92773,11699,80960,1
1,-1.791902,-0.592837,0.701794,0.587101,-1.744663,-0.753907,-0.519023,0.124541,-0.844541,-1.512335,...,0.581688,-0.635372,A,C,A123,B,74225,84859,29422,0
2,0.921972,-0.922185,-0.140028,1.150036,-0.174820,0.908685,-0.918200,1.653109,0.203502,0.388524,...,1.006870,1.577429,C,D,A123,A,40176,13468,78436,0
3,-0.483944,2.510178,-0.133058,-1.389249,-0.593578,0.692654,-0.873139,0.368663,2.298194,0.793146,...,0.839672,-0.332280,E,B,None,D,70664,46358,74595,0
4,-0.704517,-0.557355,2.117727,0.504921,-0.207584,0.115288,0.900155,0.505530,0.671575,-0.934065,...,-0.769236,-0.526820,D,B,C123,A,13563,30744,63739,0
5,0.580841,-0.824399,2.489305,0.469803,2.000615,1.053706,-1.195979,0.555427,0.215523,-0.671976,...,-1.255118,0.582932,A,D,C123,C,12864,98081,24761,0
6,0.461438,0.872091,-0.321229,-2.305961,0.121518,0.701814,0.390427,1.103220,-0.402621,-2.145772,...,-2.071655,1.275945,E,E,C123,D,55314,31763,27798,1
7,-0.775023,-0.630804,0.273734,0.306011,1.555520,1.179849,0.497596,0.872957,-1.056240,-0.074955,...,-0.314705,0.254896,E,E,C123,A,18953,86240,43405,0
8,-0.032735,-0.442915,-0.385925,-1.103285,0.179465,0.620307,-1.095026,2.410319,-0.372356,-0.277937,...,-0.854912,1.445790,B,A,B34,E,15187,33371,41796,1
9,-0.091936,-0.058692,2.845677,-0.311536,-0.487323,-0.437876,2.080178,1.795949,0.202492,-0.534802,...,0.340273,0.947724,A,A,None,A,76452,10962,44292,0


In [46]:
d2 = make_synthetic_comorbidity_data(d1)

In [47]:
d2

,SUBJECT_ID,HADM_ID,SEQ_NUM,ICD9_CODE
0,92773,11699,1,V458ed1
1,92773,11699,2,41401
2,92773,11699,3,RE51881
3,74225,84859,1,4280
4,74225,84859,2,25000
...,...,...,...,...
145,77338,91106,2,V458ed1
146,77338,91106,3,RE51881
147,64954,80019,1,4280
148,64954,80019,2,RE51881


In [ ]:

# def train_model():
#     ########################
#     ### 1. PREPROCESSING ###
#     ########################
#
#     # Load in train data
#     df = pd.read_csv(RAW_DATA_PATH)
#     print("✅ TRAINING DATA LOADED")
#
#     # Create basic features
#     df = create_basic_features(df)
#     print("✅ AGE AND QUARTER-OF-DAY CREATED")
#
#     # Create demographic feature
#     df = encoder_demographics(df)
#     print("✅ DEMOGRAPHIC FEATURES CENSORED AND INTERIM DROPPED")
#
#     #  Drop unused features
#     df = drop_features(df, DROP_FEATURES)
#     print("✅ UNUSED FEATURES DROPPED")
#
#     # Change ICD column names (extract first three chars)
#     df = change_feature_names(df)
#     print("✅ ICD9 CODES SHORTENED - TRAIN")
#
#     # Change ICD9 features (extract first three chars) from comorbidities
#     comorbidities = pd.read_csv(COMORBIDITY_DATA_PATH)
#     comorbidities = change_comorbidities_icd9code(comorbidities)
#     print("✅ ICD9 CODES SHORTENED - COMORBIDITIES")
#
#     # Create target encoding
#     # Encode ICD9 codes
#     df, mapping = encoder_icd9_codes(df, comorbidities)
#
#     print("✅ ICD9 CODE ENCODED")
#
#     # Drop ID features
#     df = df.drop(ID_FEATURES, axis = 1)
#     print("✅ ID FEATURES DROPPED")
#
#     X = df[TRAIN_NUMERICAL_FEATURES + TRAIN_CATEGORICAL_FEATURES]
#     y = df[TARGET]
#
#     ########################
#     ### 2. TRANSFORMERS ###
#     ########################
#     # Encode feature "DIAGNOSOS"
#     diagnosis_encoder = TargetEncoder(
#         cols=['DIAGNOSIS'],
#         smoothing=0.25,  # reduces overfitting
#         handle_missing='value',
#         handle_unknown='value'
#     )
#
#     numeric_transformer = Pipeline(
#         steps=[("imputer", SimpleImputer(strategy="mean")),
#                ("scaler", StandardScaler())])
#
#     categorical_transformer = Pipeline(
#         steps=[("imputer", SimpleImputer(strategy="most_frequent")),
#                ("ohe", OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
#
#     # COMBINE PREPROCESSORS
#     preprocessor = ColumnTransformer(transformers=[
#         ("num", numeric_transformer, TRAIN_NUMERICAL_FEATURES),
#         ("cat", categorical_transformer, TRAIN_CATEGORICAL_FEATURES)],
#         remainder='drop')
#
#     model = xgb.XGBClassifier(**XGB_PARAMS)
#
#     model_pipeline = Pipeline(steps=[
#         ("diagnosis_target_encoder", diagnosis_encoder),
#         ('preprocessor', preprocessor),
#         ('classifier', model)])
#     print("✅ PIPELINE CREATED")
#
#
#     ########################
#     ### 3. TRAIN MODEL ###
#     ########################
#     model_pipeline.fit(X, y)
#     print("✅ MODEL TRAINED")
#
#     # Check model error
#     y_hat_in = model_pipeline.predict(X)
#     print("✅ MODEL RUNS")
#     print(f'1. Model Accuracy:{accuracy_score(y, y_hat_in)} \n2. Model Recall:{recall_score(y, y_hat_in)} \n3. Model Precision: {precision_score(y, y_hat_in)}')
#
#     ########################
#     ### 3. SAVE MODEL ###
#     ########################
#     print("✅ SAVING MODEL")
#     model_path = f"{MODEL_OUTPUT_PATH}/model_{MODEL_VERSION}.pkl"
#     mapping_path = f"{MODEL_OUTPUT_PATH}/icd_mapping_{MODEL_VERSION}.pkl"
#     preprocessor_path = f"{MODEL_OUTPUT_PATH}/preprocessor_{MODEL_VERSION}.pkl"
#
#     joblib.dump(model_pipeline, model_path)
#     joblib.dump(mapping, mapping_path)
#     joblib.dump(preprocessor, preprocessor_path)
#     print("✅ SAVED MODEL")
#
#     return model_pipeline
#


In [ ]:



# def train_model_from_df(df, comorbidities):
#
#     ########################
#     ### 1. PREPROCESSING ####
#     ########################
#
#     # Create basic features
#     df = create_basic_features(df)
#     print("✅ AGE AND QUARTER-OF-DAY CREATED")
#
#     # Create demographic feature
#     df = encoder_demographics(df)
#     print("✅ DEMOGRAPHIC FEATURES CENSORED AND INTERIM DROPPED")
#
#     # Drop unused features
#     df = drop_features(df, DROP_FEATURES)
#     print("✅ UNUSED FEATURES DROPPED")
#
#     # Change ICD column names (extract first three chars)
#     df = change_feature_names(df)
#     print("✅ ICD9 CODES SHORTENED - TRAIN")
#
#     # Shorten comorbid ICD codes
#     comorbidities = change_comorbidities_icd9code(comorbidities)
#     print("✅ ICD9 CODES SHORTENED - COMORBIDITIES")
#
#     # Encode ICD9 comorbidities data
#     df, mapping = encoder_icd9_codes(df, comorbidities)
#     print("✅ ICD9 CODE ENCODED")
#
#     # Drop ID features
#     df = df.drop(ID_FEATURES, axis=1)
#     print("✅ ID FEATURES DROPPED")
#
#     # TRAINING INPUTS
#     X = df[TRAIN_NUMERICAL_FEATURES + TRAIN_CATEGORICAL_FEATURES]
#     y = df[TARGET]
#
#     ########################
#     ### 2. TRANSFORMERS ####
#     ########################
#     diagnosis_encoder = TargetEncoder(
#         cols=['DIAGNOSIS'],
#         smoothing=0.25,
#         handle_missing='value',
#         handle_unknown='value'
#     )
#
#     numeric_transformer = Pipeline([
#         ("imputer", SimpleImputer(strategy="mean")),
#         ("scaler", StandardScaler())
#     ])
#
#     categorical_transformer = Pipeline([
#         ("imputer", SimpleImputer(strategy="most_frequent")),
#         ("ohe", OneHotEncoder(handle_unknown='ignore', sparse_output=False))
#     ])
#
#     preprocessor = ColumnTransformer(
#         transformers=[
#             ("num", numeric_transformer, TRAIN_NUMERICAL_FEATURES),
#             ("cat", categorical_transformer, TRAIN_CATEGORICAL_FEATURES)
#         ],
#         remainder='drop'
#     )
#
#     model = xgb.XGBClassifier(**XGB_PARAMS)
#
#     model_pipeline = Pipeline([
#         ("diagnosis_target_encoder", diagnosis_encoder),
#         ("preprocessor", preprocessor),
#         ("classifier", model)
#     ])
#     print("✅ PIPELINE CREATED")
#
#     ########################
#     ### 3. TRAIN MODEL ###
#     ########################
#     model_pipeline.fit(X, y)
#     print("✅ MODEL TRAINED")
#
#     # Check model error
#     y_hat_in = model_pipeline.predict(X)
#     print("✅ MODEL RUNS")
#     print(
#         f'1. Model Accuracy:{accuracy_score(y, y_hat_in)} \n'
#         f'2. Model Recall:{recall_score(y, y_hat_in)} \n'
#         f'3. Model Precision: {precision_score(y, y_hat_in)}'
#     )
#
#     return model_pipeline, mapping, preprocessor
#
# def train_model():
#     ########################
#     ### 1. PREPROCESSING ###
#     ########################
#
#     df = pd.read_csv(RAW_DATA_PATH)
#     comorbidities = pd.read_csv(COMORBIDITY_DATA_PATH)
#     print("✅ TRAINING DATA LOADED (FROM CSV)")
#
#     model_pipeline, mapping, preprocessor = train_model_from_df(df, comorbidities)
#
#     ########################
#     ### 2. SAVE MODEL ######
#     ########################
#     print("✅ SAVING MODEL")
#     model_path = f"{MODEL_OUTPUT_PATH}/model_{MODEL_VERSION}.pkl"
#     mapping_path = f"{MODEL_OUTPUT_PATH}/icd_mapping_{MODEL_VERSION}.pkl"
#     preprocessor_path = f"{MODEL_OUTPUT_PATH}/preprocessor_{MODEL_VERSION}.pkl"
#
#     joblib.dump(model_pipeline, model_path)
#     joblib.dump(mapping, mapping_path)
#     joblib.dump(preprocessor, preprocessor_path)
#
#     print("✅ SAVED MODEL")
#

In [ ]:
def test_preprocess():
    df = make_fake_patient_data()
    df["DOB"] = random_dates("1920-01-01", "2000-12-31", 10)
    comorbidities_df = make_comorbidities_data()

    df = create_basic_features(df)
    df = encoder_demographics(df)
    df = drop_features(df, DROP_FEATURES)
    df = change_feature_names(df)
    comorbidities_df = change_comorbidities_icd9code(comorbidities_df)
    df,mapping = encoder_icd9_codes(df, comorbidities_df)

    df = df.drop(ID_FEATURES, axis = 1)

    # Assertions
    # New features created - inlcudes mortality features (enocder_demographics)
    for feature in ENGINEERED_MORTALITY_PROXIES + ENGINEERED_NUMERICAL_FEATURES + ENGINEERED_CATEGORICAL_FEATURES:
        assert feature in df.columns, f"feature {feature} present in dataframe"

    # Features dropped
    for feature in DROP_FEATURES:
        assert feature not in df.columns, f"Column {feature} was not dropped"
    # Age
    assert pd.api.types.is_numeric_dtype(df[ENG_AGE])
    assert df[ENG_AGE].between(0, 90).all()
    # Quarter-of-day
    assert df[ENG_QUART_DAY].dtype.name == "category"

    # Demographic missing
    assert set(df[ENG_DEMOGRAPHIC].unique()).issubset({0, 1})

    # Mortality proxies:
    for col in ENGINEERED_MORTALITY_PROXIES:
        assert pd.api.types.is_numeric_dtype(df[col])
    assert df[MAX_MORTALITY].between(0, 1).all()
    assert df[MEAN_MORTALITY].between(0, 1).all()
    assert df[COUNT_COMORBIDITIES].min() >= 0
    assert pd.api.types.is_integer_dtype(df[COUNT_COMORBIDITIES])

In [ ]:
def test_model_saving(trained_model_and_mapping):
    model_path, mapping_path, _ = trained_model_and_mapping

    assert model_path.exists(), "Model doesn't not exist"
    assert mapping_path.exists(), "Mapping doesn't exist"

    loaded_model = joblib.load(model_path)
    loaded_mapping = joblib.load(mapping_path)

    assert hasattr(loaded_model, "predict"), "Model cannot predict"
    assert isinstance(loaded_mapping, pd.DataFrame), "Mapping is not a dataframe"

def test_model_predict(trained_model_and_mapping):
    model_path, _, df_processed = trained_model_and_mapping

    loaded_model = joblib.load(model_path)
    X = df_processed[TRAIN_NUMERICAL_FEATURES + TRAIN_CATEGORICAL_FEATURES]

    prediction = loaded_model.predict(X)
    prediction_proba = loaded_model.predict_proba(X)[:,1]

    assert len(prediction) == len(X), "Prediction length doesn't match"
    assert set(prediction).issubset({0,1}), "Prediction values are outside valide ranges"
    assert pd.Series(prediction_proba).between(0,1).all(), "Prediction probability values are outside valide ranges"
